# __MODEL_LABEL_MARKDOWN__: champion and challenger review

Compare the current champion with published candidates and weekly challengers.
Select a package number, review its SQL summary and metrics, then
promote that reviewed package in the final cell. Review needs no local model files.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

MODEL_NAME = "__MODEL_NAME__"  # Set to None to select by label only.
MODEL_LABEL = "__MODEL_LABEL__"
DEPLOYMENT_SLOT = "__DEPLOYMENT_SLOT__"

reviewed = None  # Rerunning setup requires a new review.


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

from pricing_pipeline.notebook import (
    connect,
    deploy_model_version,
    list_challengers,
    load_registered_model,
    review_model_version,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"


## List the champion and challengers

`Model version` identifies the model's declared features, groupings, special levels and fit settings.
Weekly refits keep this number. Changed configuration gets its own model version; reusing the
same configuration reuses its model version.

`Package` identifies one saved rating result. A refit can produce another package under the
same model version. Enter that package number in `PACKAGE_VERSION` below.

`Role` tells you whether the package is the `CHAMPION`, a `CHALLENGER`, or a
`FORMER_CHAMPION` in `DEPLOYMENT_SLOT`. Promotion changes the role, not these numbers.
SQL users can read the same information in `pricing.V_MODEL_REGISTRY`.


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
deployable = list_challengers(pricing, model=model)
print(f"Saved packages in {DEPLOYMENT_SLOT}")
if deployable.empty:
    print("No published packages were found. Save a model in notebook 03 first.")
else:
    if not deployable["role"].eq("CHAMPION").any():
        print("No package is deployed in this slot.")
    display(deployable.loc[:, [
        "package_version", "definition_revision", "role", "refit_type",
        "data_as_of_date", "published_at",
    ]].rename(columns={
        "package_version": "Package",
        "definition_revision": "Model version",
        "role": "Role",
        "refit_type": "Fit",
        "data_as_of_date": "Data as of",
        "published_at": "Saved at",
    }))


## Choose a package

Find the `Package` number you want in the table above. Enter it below and run the cell.


In [ ]:
PACKAGE_VERSION = None  # Enter a Package number from the table above.
reviewed = None


## Review the selected package

The first table identifies your selected package and the current champion's package number.
The metrics table names each package and says whether it is the current champion in this slot.
"Champion used for this comparison" means the package the challenger was built against.
It can be a former champion. Check `Current champion?` to see whether it is still active.

These are the saved scores for the dates and scopes shown. The result is kept in `reviewed`
for the promotion cell. If the champion changes after this review, refresh the package list
and review again before promoting.


In [ ]:
reviewed = None
if PACKAGE_VERSION is None:
    raise ValueError(
        "Choose PACKAGE_VERSION in the cell above, run it, then rerun this review cell."
    )
if isinstance(PACKAGE_VERSION, bool) or not isinstance(PACKAGE_VERSION, int):
    raise ValueError("PACKAGE_VERSION must be an integer from the displayed published list.")
if deployable.empty:
    raise LookupError("No published candidate packages were found.")
if PACKAGE_VERSION not in set(deployable["package_version"].astype(int)):
    raise ValueError("PACKAGE_VERSION is not in the displayed published list.")
reviewed = review_model_version(
    pricing,
    model=model,
    package_version=PACKAGE_VERSION,
)
print("Selected package")
display(reviewed.summary.loc[:, [
    "package_version", "role", "definition_revision", "refit_type",
    "data_as_of_date", "current_package_version",
]].rename(columns={
    "package_version": "Package",
    "role": "Role",
    "definition_revision": "Model version",
    "refit_type": "Fit",
    "data_as_of_date": "Data as of",
    "current_package_version": "Current champion package",
}))
print("Metrics by package")
display(reviewed.metrics.loc[:, [
    "comparison", "package_version", "is_current_champion", "data_as_of_date",
    "metric_name", "metric_value", "metric_scope",
]].rename(columns={
    "comparison": "Comparison",
    "package_version": "Package",
    "is_current_champion": "Current champion?",
    "data_as_of_date": "Data as of",
    "metric_name": "Metric",
    "metric_value": "Value",
    "metric_scope": "Scope",
}))


## Promote the reviewed package

Enter the decision in the next cell after reviewing the selected package.
Then run the promotion cell. Editing this decision preserves the reviewed package.
Promotion changes the champion in `DEPLOYMENT_SLOT`. It does not rerun a fit.
Notebook 07 and scheduled monitoring create challengers without promoting them.
The expected-champion check prevents this cell from replacing a champion that changed
since review. Changing `PACKAGE_VERSION` also requires another review.


In [ ]:
DEPLOYMENT_REASON = ""  # Explain why this reviewed package should become the champion.


In [ ]:
try:
    if reviewed is None:
        raise ValueError("Run the SQL review cell before promotion.")
    if PACKAGE_VERSION != reviewed.package_version:
        raise ValueError(
            "PACKAGE_VERSION changed after review. Rerun the review cell before promotion."
        )
    if not DEPLOYMENT_REASON.strip():
        raise ValueError("Describe the approval for changing the slot's champion.")
    deployment = deploy_model_version(
        pricing,
        package=reviewed,
        reason=DEPLOYMENT_REASON,
    )
    display(deployment)
finally:
    pricing.engine.dispose()
